# Midterm Project - Comparing Three NLP Models

## Objective: To compare three langauge models, Bigram, LSTM, and Transformer on next-character prediction task.

## Dataset: Tiny Shakespeare

##Step 1: Load Dataset

In [19]:
# Load dataset

# Import urllib.request to download the dataset from a URL
import urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url, "tiny_shakespeare.txt")

# Open downloaded file using UTF-8 encoding
with open("tiny_shakespeare.txt", "r", encoding= "utf-8") as f:
    text = f.read()

# Print total number of characters in dataset
print("Total Characters:", len(text))
print("\nFirst 500 characters:\n")
# Print first 500 characters to preview dataset
print(text[:500])



Total Characters: 1115394

First 500 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


##Step 2: Character Tokenization

Goal: Convert characters into numerical IDs.

Explanation:
- Create a vocabulary and assign each unique character a numerical ID.
- Use encode() and decode() to convert between text and IDs.

In [20]:
# Character Tokenization

# Get all unique characters and sort them
chars = sorted(set(text))

# Count the number of unique characters
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print("\nCharacters:", ''.join(chars))

# Create dictinaries to convert characters to IDs and back
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
  """
  Convert text into numerical character IDs.

  Args:
    s (str): Input text to encode.

  Returns:
    list: List of numerical ID for each character
  """
  # Convert each character in text into its numerical id
  return [stoi[c] for c in s]

def decode(ids):
  """
  Convert text into numerical character IDs.

  Args:
    ids (list): List of character IDs to decode.

  Returns:
    str: Decoded text
  """
  # Convert each numerical ID back to character and join them in text
  return "".join([itos[i] for i in ids])

# Test the encoding and decoding functions
sample = "hello"
encoded = encode(sample)
decoded = decode(encoded)

print("Original:", sample)
print("Encoded:", encoded)
print("Decoded:", decoded)





Vocabulary size: 65

Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Original: hello
Encoded: [46, 43, 50, 50, 53]
Decoded: hello


## Step 3: Training and Validation Split
Convert the dataset into numerical IDs and split 90% to training data and 10% validation data.

In [21]:
# Step 3: Training and Validation Split

import torch

# Convert entire dataset into numerical IDs
data = torch.tensor(encode(text),dtype=torch.long)

# Calcualte the index for 90% of the dataset
n = int(0.9 * len(data))

# Split data into training and validation sets
train_data = data[:n]
val_data = data[n:]

# Print the size of each dataset
print("Training characters:", len(train_data))
print("Validation characters:", len(val_data))

# Check first 20 training IDs
print("First 20 training IDs:", train_data[:20])

Training characters: 1003854
Validation characters: 111540
First 20 training IDs: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])


## Step 4: Create Input and Target Sequences

Goal: Create input and target sequences for next-character prediction.

Explanation:
- The input contains previous characters.
- The target contains next character at each position.

In [22]:
# Set maximum context length
block_size = 32

def get_batch(data, batch_size = 32):
  """
  Create a batch of input and target sequences.

  Args:
    data (torch.Tensor): Training or validation data.
    batch_size (int): Number of sequences in each batch.

  Returns:
    tuple: Input (x) and target (y) tensors.
  """
  # Randomly select starting positions
  ix = torch.randint(len(data) - block_size, (batch_size,))

  # Create empty lists for inputs and targets
  inputs = []
  targets = []

  # Get input sequences and their next-character targets
  for i in ix:
    inputs.append(data[i:i+block_size])
    targets.append(data[i+1:i+block_size+1])

  # Combine sequences into tensors
  inputs = torch.stack(inputs)
  targets = torch.stack(targets)

  return inputs, targets

# Test function using training data
x, y = get_batch(train_data)

print("Input shape:", x.shape)
print("Target shape:", y.shape)

# View one input and its corresponding target
print("Input:", decode(x[0].tolist()))
print("Target:", decode(y[0].tolist()))



Input shape: torch.Size([32, 32])
Target shape: torch.Size([32, 32])
Input: l we serve heaven
With less resp
Target:  we serve heaven
With less respe


# Model 1: Bigram Model

# Step 5: Build a Bigram Model
Goal: Build a bigram model to predict next character

Explanation:
The Bigram model predicts the next character by using the immediate previous character. This is done as it calculates the probability of the next character based on how frequently each pair of characters appear in the training dataset. Its main advantage is that it is effective at identifying common character patterns without requiring complex training . However, its disadvantage is that it ignores all characters before the current which doesn’t allow it to capture long relationships between words and sentences, hindering its ability to understand context and generate meaningful context.


In [23]:
import time

start_time = time.perf_counter()
# Create an empty table to count character pairs
bigram_counts = torch.zeros((vocab_size, vocab_size))

# Iterate through each character in training data
# Skip iterating through last because it has no next character
for i in range(len(train_data)-1):
  # Get current character's numerical ID
  current_char = train_data[i].item()
  # Get next character's numerical ID
  next_char = train_data[i+1].item()

  # Increase count of character by pair 1
  bigram_counts[current_char, next_char] += 1

# Add 1 to all count to avoid zero probability
bigram_prob = bigram_counts + 1

# Divide each count by its row total to calculate probabilities
bigram_prob = bigram_prob / bigram_prob.sum(dim=1, keepdim=True)

# Record Bigram training time
end_time = time.perf_counter()
bigram_time = end_time - start_time
print("Bigram Training Time:", round(bigram_time, 4), "seconds")

# Print dimensions of probability table
print("Bigram probability table shape: ", bigram_prob.shape)
# Print probabilities of each possible character following 's'
print("Probabilities after 's':", bigram_prob[stoi["s"]])


Bigram Training Time: 33.9731 seconds
Bigram probability table shape:  torch.Size([65, 65])
Probabilities after 's': tensor([1.8469e-02, 3.1162e-01, 6.0822e-03, 2.2279e-05, 2.2279e-05, 4.9237e-03,
        4.8702e-02, 1.0917e-03, 1.8180e-02, 2.2279e-05, 9.5132e-03, 9.9142e-03,
        6.2382e-03, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05,
        2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05,
        2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05,
        2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05, 2.2279e-05,
        2.2279e-05, 2.2279e-05, 2.2279e-05, 2.7225e-02, 3.0522e-03, 9.9588e-03,
        1.3145e-03, 1.1220e-01, 1.0917e-03, 6.9065e-04, 6.2448e-02, 4.0704e-02,
        4.4558e-05, 2.4507e-03, 8.0428e-03, 3.9880e-03, 5.5698e-04, 5.6879e-02,
        2.3014e-02, 3.3419e-04, 4.4558e-05, 3.6115e-02, 1.3537e-01, 2.5755e-02,
        6.6837e-05, 1.1808e-02, 2.2279e-05, 1.4259e-03, 2.2279e-05])


## Step 6: Bigram Model Evaluation

Goal: Evaluate the Bigram model using training and validation loss

Explanation:
- Training loss measures how well the model predicts training data
- Validation loss measures how well the model predicts the validation data

In [24]:

def bigram_loss(data):
    """
    Calculate the average loss of the Bigram model.

    Args:
       data (torch.Tensor): Training or validation data.

    Returns:
       float: Average loss of the Bigram model.
    """
    # Current characters are the inputs
    x = data[:-1]
    # Next characters are the correct targets
    y = data[1:]

    # Get probabilities for each input characters
    probabilities = bigram_prob[x]

    # Calculate the negative log of each correct character
    losses = -torch.log(probabilities[torch.arange(len(y)), y])

    # Calculate the average loss
    average_loss = losses.mean()

    return average_loss.item()

# Calculate losses for training and validation data
train_loss = bigram_loss(train_data)
val_loss = bigram_loss(val_data)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

print("Bigram Training Loss:", round(train_loss, 4))
print("Bigram Validation Loss:", round(val_loss, 4))



Training loss: 2.45457124710083
Validation loss: 2.481889486312866
Bigram Training Loss: 2.4546
Bigram Validation Loss: 2.4819


## Step 7: Generate Text with Bigram Model

Goal: Generate text using the trained Bigram model

Explanation:
- Start with one character and predict the next character.
- Use the predicted character to continue generating text.
- Evaluate whether the text the model generated resembles Shakespeare's writing

In [25]:
def generate_bigram(start_char, length=200):
  """
    Generate text using the Bigram model.

    Args:
       start_char (str): Starting character for text generation.
       length (int): Number of characters to generate.

    Returns:
       str: Generated text.
  """
  # Start with given character
  generated = start_char
  # Convert the starting character into numerical ID
  current_char = stoi[start_char]

  # Iterate through a character at a time
  for i in range(length):
    # Get probabilities of the next character
    probs = bigram_prob[current_char]

    # Randomly select a character
    next_char = torch.multinomial(probs, 1).item()

    # Convert selected ID into a character
    generated += itos[next_char]

    # Update the current character for the next prediction
    current_char = next_char

  return generated

# Test Bigram model

# Generate 200 characters starting with 's'
sample_bigram = generate_bigram("s", 200)

# Print the generated text
print("Bigram Generated Text:")
print(sample_bigram)

Bigram Generated Text:
s the, shanghestheroole aitinehes hintoe Pis coulo tarofou alt
keld,
CKI engairye half t.

Gonde t ceeur Y my-w we wh woty ppry

Lofthe:
TEThit isabrdatash, lu
Mothy ce fte
NTounsod wert,
ELIUKEYe ms c


# Model 2: LSTM *Model*

## Step 8: Build the LSTM Model
Goal: Build an LSTM model for next-character prediction

The LSTM model predicts the next character by using information from multiple previous characters by processing characters sequentially and using memory cells and gates which determine how to remember and forget. Its main advantage is that it can capture longer patterns and relationships between characters, allowing it to generate recognizable words and sentence-like structure. However, its main disadvantage is that it can struggle with long range dependencies so the characters have to pass through multiple sequential steps, which can cause it to lose important information. This may cause the model to lose relationship and structure between distance characters and maintain meaningful sentence structures.


In [26]:
import torch.nn as nn

class LSTMModel(nn.Module):
  """
  LSTM model for next-character prediction.

  Args:
    vocab_size (int): Number of unique characters.
    embedding_dim (int): Size of character embeddings.
    hidden_size (int): Number of LSTM hidden units.
  """

  def __init__(self, vocab_size, embedding_dim=64, hidden_size=128):
    super(LSTMModel, self).__init__()

    # Convert character IDs into learned numerical vectors
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    # Process character sequence using LSTM
    self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
    # Convert LSTM outputs into predictions for each character
    self.fc = nn.Linear(hidden_size, vocab_size)

  def forward(self, x):
    """
    Predict the next character for each input position.

    Args:
      x (torch.Tensor): Batch of input character IDs.

    Returns:
      torch.Tensor: Prediction scores for each character.
    """
    # Convert input IDs into embeddings
    x = self.embedding(x)

    # Pass embeddings through LSTM
    x, _ = self.lstm(x)

    # Convert LSTM outputs into predictions
    x = self.fc(x)

    return x

# Create LSTM model
lstm_model = LSTMModel(vocab_size)


print(lstm_model)

LSTMModel(
  (embedding): Embedding(65, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=65, bias=True)
)


## Step 9: LSTM Training

Goal: Train the LSTM model to predict the next character

Explanation:
- Use batches of training data the model.
- Calculate prediction error using cross-entropy loss.
- Update model weights using Adam optimizer


In [27]:
import time
import torch.nn.functional as F

# Set learning rate and number of training steps
learning_rate = 0.001
num_steps = 500

# Set up optimizer to update model weights
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=learning_rate)

# Measure LSTM Training Time
start_time = time.perf_counter()

# Set model to training mode
lstm_model.train()

# Train the model
for step in range(num_steps):
  # Get a batch of training data and target sequence
  x, y = get_batch(train_data)

  # Get predictions from LSTM
  logits = lstm_model(x)

  # Calculate cross-entropy loss
  loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

  # Clear previous gradients
  optimizer.zero_grad()

  # Calculate gradients
  loss.backward()

  # Update model weights
  optimizer.step()

  # Print loss every 100 steps
  if (step + 1) % 100 == 0:
    print("Step:", step + 1, "Loss:", round(loss.item(), 4))

# Record ending time
end_time = time.perf_counter()

# Calculate total training time in seconds
lstm_time = end_time - start_time
print("LSTM Training Time:", round(lstm_time, 4), "seconds")

Step: 100 Loss: 2.6551
Step: 200 Loss: 2.3253
Step: 300 Loss: 2.2146
Step: 400 Loss: 2.1626
Step: 500 Loss: 2.0331
LSTM Training Time: 14.5753 seconds


## Step 10: LSTM Model Evaluation

Goal: Calculate the training and validation loss of the LSTM model.

Explanation:
- Evaluate the LSTM using training and validation datasets
- Use cross-entropy loss to measure prediction errors
- Compare LSTM losses with Bigram Model

In [28]:
def evaluate_lstm(data, num_batches=20):
  """
  Calculate the average loss of the LSTM model.

  Args:
    data (torch.Tensor): Training or validation data.
    num_batches (int): Number of batches to evaluate

  Returns:
    float: Average cross-entropy loss.
  """

  # Set model to evaluation mode
  lstm_model.eval()

  # Create an empty list to store losses
  losses = []

  # Disable gradient calculation during eval
  with torch.no_grad():
    # Iterate through batches
    for _ in range(num_batches):
      # Get input and target sequences
      x, y = get_batch(data)
      # Get predictions from LSTM
      logits = lstm_model(x)

      # Calculate cross-entropy loss
      loss = F.cross_entropy(
          logits.reshape(-1, vocab_size), y.reshape(-1)
      )

      # Store the loss
      losses.append(loss.item())
  # Calculate the average loss
  average_loss = sum(losses) / len(losses)

  return average_loss

# Evaluate both datasets
lstm_train_loss = evaluate_lstm(train_data)
lstm_val_loss = evaluate_lstm(val_data)

print("LSTM Training Loss:", round(lstm_train_loss, 4))
print("LSTM Validation Loss:", round(lstm_val_loss, 4))



LSTM Training Loss: 2.0615
LSTM Validation Loss: 2.1149


## Step 11: Generate Text with LSTM Model

Goal: Generate text using the trained LSTM Model.

Explanation:
- Start with the same character used for the Bigram model.
- Predict and generate one character at a time.
- Compare the generated text with the Bigram model.

In [29]:
def generate_lstm(start_char, length=200):
  """
  Generate text using the trained LSTM model.

  Args:
    start_char (str): Starting character.
    length (int): Number of new characters to generate.

  Returns:
    str: Generated text.
  """

  # Set model to evaluation mode
  lstm_model.eval()

  # Start with given character
  generated = start_char

  # Convert starting character into numerical ID
  current_char = stoi[start_char]

  # Create the input tensor
  x = torch.tensor([[current_char]], dtype=torch.long)

  # Generate one character at a time
  with torch.no_grad():
    for i in range(length):
      # Get predictions from LSTM
      logits = lstm_model(x)
      # Get predictions for the last character
      last_logits = logits[:,-1,:]
      # Convert prediction scores into probabilities
      probs = F.softmax(last_logits, dim=1)
      # Randomly select a character
      next_char = torch.multinomial(probs, 1).item()
      # Add predicted character to the text
      generated += itos[next_char]

      # Add the predicted character ID to the input
      next_input = torch.tensor([[next_char]], dtype=torch.long)
      x = torch.cat((x, next_input), dim=1)

      # Keep only the recent 32 characters
      x = x[:, -block_size:]

  return generated

# Generate 200 new characters starting with 's'
sample_lstm = generate_lstm("s", 200)

print("LSTM Generated Text:")
print(sample_lstm)

LSTM Generated Text:
save eritioh

Bmay youm;
And cult Mouth here serce. I lit not call absure in?

Nivee:
Ge, or hibe? ow eptajus, uand; ap the nour wiwn
Karbis pasor:
And Ther net Yirges, you muves!,
Ait theur me 'em.

L


# Model 3: Small Transformer Model

## Step 12: Small Transformer Model

Goal: Build a small Transformer model for next-character prediction.

Explanation:
The Transformer model predicts the next character by using a self-attention mechanism which examines relationships between multiple previous characters within a context window. This is done by first converting characters into numerical vectors through token embedding and using positional embeddings to represent their order in sequence. The self-attention mechanism calculates how relevant previous characters are when predicting the next character. Furthermore, it uses causal masking which blocks attention to future positions to prevent the model from accessing future characters in order to ensure that prediction is based on current and previous characters. The feedforward layer processes information and the output layer produces scores for the next possible character. These processes allow it to identify relationships between distant characters and process multiple positions in parallel during training which can improve computational efficiency and ability to learn complex language patterns.

In [30]:
import torch.nn as nn

class TransformerModel(nn.Module):
  """
  Small Transformer model for next-character prediction

  Args:
    vocab_size(int): Number of unique characters.
    embedding_dim(int): Size of character embeddings.
    num_heads (int): Number of attention heads.

  """
  def __init__(self, vocab_size, embedding_dim=64, num_heads=4):
    super().__init__()

    # Convert character IDs into embeddings
    self.embedding = nn.Embedding(vocab_size, embedding_dim)

    # Create embeddings for character positions
    self.position_embedding = nn.Embedding(block_size, embedding_dim)


    # Create self-attention layer
    self.attention = nn.MultiheadAttention(embedding_dim, num_heads,
                                           batch_first = True)

    # Feedforward layer
    self.fc1 = nn.Linear(embedding_dim, 128)
    self.fc2 = nn.Linear(128, embedding_dim)

    # Output layer for predicting next characters
    self.output = nn.Linear(embedding_dim, vocab_size)

  def forward(self, x):
    """
    Predict the next character for each input position.

    Args:
      x (torch.Tensor): Batch of input character IDs.

    Returns:
      torch.Tensor: Prediction scores for each character.
    """
    # Get sequence length
    seq_len = x.shape[1]
    # Create numerical positions
    positions = torch.arange(seq_len, device=x.device)

    # Combine character and positional embeddings
    x = self.embedding(x) + self.position_embedding(positions)

    # Prevent attention from looking at future characters
    mask = torch.triu(
        torch.ones((seq_len, seq_len), device=x.device), diagonal=1
    ).bool()

    # Apply causal self-attention
    attention_output, _ = self.attention(
        x, x, x, attn_mask=mask, need_weights=False)

    # Add attention output to original input
    x = x + attention_output

    # Apply feedforward layers
    feedforward_output = self.fc1(x)
    feedforward_output = F.relu(feedforward_output)
    feedforward_output = self.fc2(feedforward_output)

    # Add feedforward output to original input
    x = x + feedforward_output

    # Predict next character
    logits = self.output(x)

    return logits

# Create Transformer model
transformer_model = TransformerModel(vocab_size)

print(transformer_model)


TransformerModel(
  (embedding): Embedding(65, 64)
  (position_embedding): Embedding(32, 64)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=65, bias=True)
)


## Step 13: Transformer Training

Goal: Train the Transformer model to predict the next character

Explanation:
- Calculate prediction errors using cross-entropy loss
- Update model weights using Adam optimizer

In [31]:
# Set learning rate and number of training steps
learning_rate = 0.001
num_steps = 500

# Set up optimizer to update model weights
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=learning_rate)

# Record starting time
start_time = time.perf_counter()

# Set model to training mode
transformer_model.train()

# Train the model
for step in range(num_steps):
  # Get a batch of training data and target sequence
  x, y = get_batch(train_data)

  # Get predictions from Transformer
  logits = transformer_model(x)

  # Calculate cross-entropy loss
  loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

  # Clear previous gradients
  optimizer.zero_grad()

  # Calculate gradients
  loss.backward()

  # Update model weights
  optimizer.step()

  # Print loss every 100 steps
  if (step + 1) % 100 == 0:
    print("Step:", step + 1, "Loss:", round(loss.item(), 4))

# Record ending time
end_time = time.perf_counter()

# Calculate total training time in seconds
transformer_time = end_time - start_time
print("Transformer Training Time:", round(transformer_time, 4), "seconds")

Step: 100 Loss: 2.6676
Step: 200 Loss: 2.5589
Step: 300 Loss: 2.4027
Step: 400 Loss: 2.3957
Step: 500 Loss: 2.2422
Transformer Training Time: 6.8507 seconds


## Step 14: Transformer Model Evaluation

Goal: Calculate training and validation loss of the Transformer model.

Explanation:
- Evaluate the Transformer using training and validation data.
- Calculate prediction errors using cross-entropy loss

In [32]:
def evaluate_transformer(data, num_batches=20):
  """
  Calculate the average loss of the Transformer model.

  Args:
    data (torch.Tensor): Training or validation data.
    num_batches (int): Number of batches to evaluate

  Returns:
    float: Average cross-entropy loss.
  """
  # Set model to evaluation mode
  transformer_model.eval()

  # Create an empty list to store losses
  losses = []

  # Disable gradient calculation during eval
  with torch.no_grad():
    # Iterate through batches
    for i in range(num_batches):
      # Get input and target sequences
      x, y = get_batch(data)
      # Get predictions from Transformer
      logits = transformer_model(x)

      # Calculate cross-entropy loss
      loss = F.cross_entropy(
          logits.reshape(-1, vocab_size),
          y.reshape(-1)
      )

      # Store loss
      losses.append(loss.item())

  # Calculate the average loss
  average_loss = sum(losses) / len(losses)

  return average_loss

# Evaluate training and validation data
transformer_train_loss = evaluate_transformer(train_data)
transformer_val_loss = evaluate_transformer(val_data)

print("Transformer Training Loss:", round(transformer_train_loss, 4))
print("Transformer Validation Loss:", round(transformer_val_loss, 4))

Transformer Training Loss: 2.3049
Transformer Validation Loss: 2.333


## Step 15: Generate Text with Transformer Model

Goal: Generate text using the trained Transformer model.

Explanation:
- Predict one character at a time using self-attention.

In [33]:
def generate_transformer(start_char, length=200):
  """
  Generate text using the trained Transformer model.

  Args:
    start_char (str): Starting character.
    length (int): Number of new characters to generate.

  Returns:
    str: Generated_text.
  """

  # Set model to evaluation mode
  transformer_model.eval()

  # Start with given character
  generated = start_char

  # Convert the starting character into a numerical ID
  current_char = stoi[start_char]

  # Create the input tensor
  x = torch.tensor([[current_char]], dtype=torch.long)

  # Generate one character at a time
  with torch.no_grad():
    for i in range(length):
      # Keep only most recent 32 characters
      x = x[:, -block_size:]

      # Get predictions from Transformer
      logits = transformer_model(x)
      # Get predictions for the last character
      last_logits = logits[:,-1,:]

      # Convert prediction scores into probabilities
      probs = F.softmax(last_logits, dim=1)

      # Randomly select a character
      next_char = torch.multinomial(probs, 1).item()

      # Add the predicted character to the text
      generated += itos[next_char]

      # Add the predicted character ID to the input
      next_input = torch.tensor([[next_char]], dtype=torch.long)
      x = torch.cat((x, next_input), dim=1)

  return generated

# Generate 200 new characters starting with 's'
sample_transformer = generate_transformer("s", 200)

print("Transformer Generated Text:")
print(sample_transformer)


Transformer Generated Text:
s pink,
BANVENVOigh. INCMTlar FRIAUSot Won
NA Ten saillerd duesestet, Lold qoste, fareerouthet.

Anter meand Kamlfoxtt imushreno hat of tien! conen theat glenof wo qos thim live, not toe of ak'd wief b


## Step 16: Create Consistent Evaluation Batches

Goal: Prepare the same evaluation data for all three models.

Explanation:
- Use a random seed to create consistent training and validation batches.
- Test all three models on the same sequences for fair comparison

In [34]:
# Set random seed for consistent results
torch.manual_seed(42)

# Create empty lists for evaluation batches
train_batches = []
val_batches = []

# Create 20 batches for each dataset
for i in range(20):
  train_batches.append(get_batch(train_data))
  val_batches.append(get_batch(val_data))

print("Training batches:", len(train_batches))
print("Validation batches:", len(val_batches))

Training batches: 20
Validation batches: 20


## Step 17: Compare Model Losses

Goal: Calculate the training and validation losses for all three models using the same evaluation batches.

Explanation:
- Evaluate Bigram, LSTM, and Transformer on identical data.
- Use cross-entropy loss for all three models.
- Compare which model has the lowest loss.


In [35]:
def evaluatemodels(model_name, batches):
  """
  Calculate average loss for a model using fixed batches.
  Args:
    model_name(str): Name of the model to evaluate.
    batches (list): List of input and target batches

  Returns:
    float: Average cross-entropy loss.
  """
  losses = []

  # Set neural network models to evaluation mode
  lstm_model.eval()
  transformer_model.eval()

  # Disable gradient calculations
  with torch.no_grad():
    for x, y in batches:
      # Evaluate Bigram model
      if model_name == "Bigram":
        probs = bigram_prob[x]
        # Get the probability of each correct target
        correct_probs = probs.gather(
            2, y.unsqueeze(2)).squeeze(2)

        # Caclulate Bigram loss
        loss = -torch.log(correct_probs).mean()

      elif model_name == "LSTM":
        logits = lstm_model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size), y.reshape(-1)
        )

      elif model_name == "Transformer":
        # Get Transformer predictions
        logits = transformer_model(x)
        # Calculate cross-entropy loss
        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1)
        )
      # Store the loss
      losses.append(loss.item())
  # Calculate average loss across all batches
  average_loss = sum(losses) / len(losses)

  return average_loss


# Calculate training losses
bigram_train_loss = evaluatemodels("Bigram", train_batches)
lstm_train_loss = evaluatemodels("LSTM", train_batches)
transformer_train_loss = evaluatemodels("Transformer", train_batches)

# Calculate validation losses
bigram_val_loss = evaluatemodels("Bigram", val_batches)
lstm_val_loss = evaluatemodels("LSTM", val_batches)
transformer_val_loss = evaluatemodels("Transformer", val_batches)

# Print results
print("Bigram:", round(bigram_train_loss, 4), round(bigram_val_loss, 4))
print("LSTM:", round(lstm_train_loss, 4), round(lstm_val_loss, 4))
print("Transformer:", round(transformer_train_loss, 4), round(transformer_val_loss, 4))




Bigram: 2.4512 2.5005
LSTM: 2.0705 2.1103
Transformer: 2.314 2.35


## Step 18: Compare Model Performance

Goal: Compare training loss, validation loss and training time of all models.



In [36]:
import pandas as pd

# Store results for all models
results = {
    "Model": ["Bigram", "LSTM", "Transformer"],
    "Training Loss": [bigram_train_loss, lstm_train_loss, transformer_train_loss],
    "Validation Loss": [bigram_val_loss, lstm_val_loss, transformer_val_loss],
    "Training Time": [bigram_time, lstm_time, transformer_time]
}

# Create a table
results_df = pd.DataFrame(results)
results_df = results_df.round(4)
display(results_df)


,Model,Training Loss,Validation Loss,Training Time
0,Bigram,2.4512,2.5005,33.9731
1,LSTM,2.0705,2.1103,14.5753
2,Transformer,2.3140,2.3500,6.8507


## Step 19: Compare Generated Text

Goal: Compare the quality of text generated by Bigram, LSTM and Transformer models.

Explanation:
- Examine which models generates more recognizable words, sentences and dialogues


| Model | Generated Text Sample | Generated Text Quality | Observation |
|---|---|---|---|
| Bigram | `s the, shanghestheroole aitinehes hintoe Pis coulo` | Low | Mostly random characters with some recognizable word patterns and punctuation. |
| LSTM | `save eritioh` | Medium | Some recognizable words and dialogue-like formatting, but still difficult to understand. |
| Transformer | `s pink,` | Low | Some recognizable words, but many incomplete or nonsensical words. |


## Reflection

Overall, the model which performed the best based on achieving lowest training and validation losses and generating greater recognizable words and sentence like structure was
 LSTM. Since my dataset, Tiny Shakespeare, contains dialogue, repeated language patterns and relationships between characters across sequences, LSTM was able to use its function of remembering important information from previous characters and use it to predict the next character through memory cells. These memory cells allow the model to decide which information to remember or forget, which allows them to learn longer patterns and generate sentence-like structure and create recognizable words. Comparatively, Bigram did not perform well resulting in mostly random characters with few recognizable words as it only relies on previous characters.
However, the Transformer model performed better than the Bigram model but not LSTM. This is because while LSTM processes character sequentially which allows it to maintain a memory state, the Transformer uses self-attention which it can use to examine multiple relationships within a context window. Self-attention occurs by calculating attention score which helps determine how relevant each previous character is to current character, increasing its ability to understand context. Thus, Transformers can be effective when the dataset consists of longer sequences of text because the self-attention mechanism helps them identify relationships between distant characters and capture contextual information.In terms of training time, the Transformer was the fastest at 6.85 seconds, followed by the LSTM at 14.58 seconds and the Bigram at 33.97 seconds. Even though the Transformer trained the fastest and performed better than the Bigram, its complexity didnt result in increased accuracy.
It is possible that in my case, Transformer performed poorer to LSTM due to limitations of small model architecture, short context length of 32 characters and limited training could have prevented the model from learning complex patterns and relationship between characters in the dataset.
Some other limitations of this experiment are that all three models use character-level prediction instead of world-level prediction. This means that all three models had to learn how individual characters had to combine to form words before learning language patterns and sentence structure which increased the likelihood of incomplete words.
Furthermore, another limitation was that the evaluation heavily relied on cross-entropy loss and a small number of generated text samples which does not accurately represent how effective these models would be in real life. Even though cross-entropy loss does measure how accurately a model predicts the next character, it doesnt show whether the model can generate meaningful words, correct sentence structure and dialogue. Additionally, presenting limited general text samples could be a biased conclusion as quality of each models output can vary so it wouldnt be a reliable judgement to conclude on its overall text generation quality.



## What I Learned

I was actually surprised about the results as I was expecting Transformers to perform better as it is a more complex model and uses a self-attention mechanism which in theory allows it to capture relationships between distant characters effectively. Since LSTMs process characters sequentially, I assumed that its reliance on memory state would limit their ability to capture relationships between distant characters. However, through this experiment I learned that factors such as dataset size, context length, model architecture and training configuration can significantly impact a models performance as they determine how much information the model can learn, capture relationships between characters and make accurate predictions.